# LABORATORIO N° 08
## Semana 8: ML Supervisado - Clasificación

**Curso:** Fundamentos de Gestión de Datos  
**Docente:** Pilar Rocío Sayán Mejía  
**Modalidad:** Individual  
**Nombre del estudiante:**  
**Sección:**  
**Fecha:**  

### Caso de negocio
Una entidad financiera desea identificar qué clientes tienen mayor riesgo de caer en mora el siguiente mes. Para ello se usa el dataset real **Default of Credit Card Clients** de UCI. La variable objetivo está representada como **0 = no cae en mora** y **1 = sí cae en mora**.

### Fundamento teórico

| N° | Concepto / Principio | Definición aplicada al caso |
| --- | --- | --- |
| 1 | Clasificación | Tarea de aprendizaje supervisado donde el resultado pertenece a una categoría. |
| 2 | Variable objetivo | Columna que se desea predecir; en este caso, mora_siguiente_mes. |
| 3 | Clase 0 | Cliente que no cae en mora el siguiente mes. |
| 4 | Clase 1 | Cliente que sí cae en mora el siguiente mes. |
| 5 | Árbol de decisión | Modelo que clasifica mediante reglas tipo si/entonces. |
| 6 | Matriz de confusión | Tabla que compara clases reales contra clases predichas. |
| 7 | Accuracy | Proporción total de aciertos del modelo. |
| 8 | Precision | De los clientes predichos como mora, cuántos realmente cayeron en mora. |
| 9 | Recall | De los clientes que realmente cayeron en mora, cuántos detectó el modelo. |
| 10 | F1-score | Promedio balanceado entre precision y recall. |

## Actividad 1: Carga del dataset real

In [ ]:
# Paso 1: cargar librerías y descargar el dataset real
import sys
import subprocess
import pandas as pd
import numpy as np

try:
    import matplotlib.pyplot as plt
    import seaborn as sns
    HAS_PLOTS = True
except Exception:
    HAS_PLOTS = False

try:
    from sklearn.model_selection import train_test_split
    from sklearn.tree import DecisionTreeClassifier, plot_tree
    from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "scikit-learn"])
    from sklearn.model_selection import train_test_split
    from sklearn.tree import DecisionTreeClassifier, plot_tree
    from sklearn.metrics import confusion_matrix, classification_report, accuracy_score, precision_score, recall_score, f1_score

try:
    from IPython.display import display
except Exception:
    display = print

if HAS_PLOTS:
    sns.set_theme(style="whitegrid")

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", "{:.4f}".format)

fuente_uci = "Default of Credit Card Clients - UCI Machine Learning Repository, ID 350"
url_respaldo = "https://huggingface.co/datasets/scikit-learn/credit-card-clients/resolve/main/UCI_Credit_Card.csv"

try:
    from ucimlrepo import fetch_ucirepo
except Exception:
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "ucimlrepo"])
    from ucimlrepo import fetch_ucirepo

try:
    datos_uci = fetch_ucirepo(id=350)
    X_original = datos_uci.data.features
    y_original = datos_uci.data.targets
    df_original = pd.concat([X_original, y_original], axis=1)
    origen_carga = fuente_uci
except Exception as error_uci:
    print("No se pudo acceder directamente a UCI en este entorno.")
    print("Se usará una copia pública del mismo dataset para continuar el laboratorio.")
    print("Detalle técnico:", error_uci)
    df_original = pd.read_csv(url_respaldo)
    origen_carga = "Copia pública del dataset UCI_Credit_Card.csv"

print("Fuente original:", fuente_uci)
print("Origen usado en esta ejecución:", origen_carga)
print("Dimensiones originales:", df_original.shape)
display(df_original.head())


| P1 | ¿De dónde se obtiene el dataset y qué problema de negocio representa? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

## Actividad 2: Columnas en español y variables del modelo

In [ ]:
# Paso 2: traducir nombres de columnas y seleccionar variables sencillas
mapa_columnas = {
    "X1": "limite_credito",
    "X2": "sexo",
    "X3": "educacion",
    "X4": "estado_civil",
    "X5": "edad",
    "X6": "estado_pago_septiembre",
    "X7": "estado_pago_agosto",
    "X8": "estado_pago_julio",
    "X12": "factura_septiembre",
    "X18": "pago_septiembre",
    "Y": "mora_siguiente_mes",
    "LIMIT_BAL": "limite_credito",
    "SEX": "sexo",
    "EDUCATION": "educacion",
    "MARRIAGE": "estado_civil",
    "AGE": "edad",
    "PAY_0": "estado_pago_septiembre",
    "PAY_2": "estado_pago_agosto",
    "PAY_3": "estado_pago_julio",
    "BILL_AMT1": "factura_septiembre",
    "PAY_AMT1": "pago_septiembre",
    "default.payment.next.month": "mora_siguiente_mes",
}

df = df_original.rename(columns=mapa_columnas).copy()

columnas_modelo = [
    "limite_credito",
    "edad",
    "estado_pago_septiembre",
    "estado_pago_agosto",
    "estado_pago_julio",
    "factura_septiembre",
    "pago_septiembre",
    "mora_siguiente_mes",
]

faltantes = [col for col in columnas_modelo if col not in df.columns]
if faltantes:
    raise ValueError(f"Faltan columnas esperadas: {faltantes}")

df_modelo = df[columnas_modelo].dropna().copy()
df_modelo["mora_siguiente_mes"] = df_modelo["mora_siguiente_mes"].astype(int)

print("Columnas usadas en español:")
display(pd.DataFrame({"columna": columnas_modelo}))
print("Dimensiones del dataset para el modelo:", df_modelo.shape)
display(df_modelo.head())


| P2 | ¿Cuál es la variable objetivo y qué significa? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

| P3 | Menciona tres variables predictoras usadas para clasificar el riesgo de mora. |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

## Actividad 3: Representación de las clases

In [ ]:
# Paso 3: revisar cómo está representada la variable objetivo
resumen_objetivo = (
    df_modelo["mora_siguiente_mes"]
    .value_counts()
    .rename_axis("mora_siguiente_mes")
    .reset_index(name="cantidad")
)
resumen_objetivo["porcentaje"] = resumen_objetivo["cantidad"] / len(df_modelo) * 100
resumen_objetivo["interpretacion"] = resumen_objetivo["mora_siguiente_mes"].map({
    0: "No cae en mora el siguiente mes",
    1: "Sí cae en mora el siguiente mes"
})

display(resumen_objetivo)

if HAS_PLOTS:
    plt.figure(figsize=(6, 4))
    sns.countplot(data=df_modelo, x="mora_siguiente_mes", hue="mora_siguiente_mes", palette=["#6AA84F", "#CC4125"], legend=False)
    plt.xticks([0, 1], ["0: no mora", "1: mora"])
    plt.title("Distribución de la variable objetivo")
    plt.xlabel("mora_siguiente_mes")
    plt.ylabel("Cantidad de clientes")
    plt.show()


| P4 | ¿Cómo está representada la clase 0 y cómo está representada la clase 1? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

| P5 | ¿La base está balanceada o hay más clientes de una clase? Justifica con porcentajes. |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

## Actividad 4: Entrenamiento del árbol de decisión

In [ ]:
# Paso 4: separar entrenamiento/prueba y entrenar un árbol de decisión
variables_predictoras = [
    "limite_credito",
    "edad",
    "estado_pago_septiembre",
    "estado_pago_agosto",
    "estado_pago_julio",
    "factura_septiembre",
    "pago_septiembre",
]

X = df_modelo[variables_predictoras]
y = df_modelo["mora_siguiente_mes"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=42,
    stratify=y
)

modelo_arbol = DecisionTreeClassifier(
    max_depth=4,
    min_samples_leaf=80,
    class_weight="balanced",
    random_state=42
)

modelo_arbol.fit(X_train, y_train)

print("Filas de entrenamiento:", X_train.shape[0])
print("Filas de prueba:", X_test.shape[0])
print("Modelo entrenado: Árbol de Decisión")


| P6 | ¿Por qué se separa la base en entrenamiento y prueba? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

| P7 | ¿Qué ventaja tiene usar un árbol de decisión como primer modelo de clasificación? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

## Actividad 5: Evaluación del modelo

In [ ]:
# Paso 5: evaluar el modelo de clasificación
y_pred = modelo_arbol.predict(X_test)

matriz = confusion_matrix(y_test, y_pred)
metricas = pd.DataFrame({
    "métrica": ["accuracy", "precision", "recall", "f1_score"],
    "valor": [
        accuracy_score(y_test, y_pred),
        precision_score(y_test, y_pred),
        recall_score(y_test, y_pred),
        f1_score(y_test, y_pred),
    ]
})

print("Matriz de confusión")
display(pd.DataFrame(
    matriz,
    index=["Real: no mora", "Real: mora"],
    columns=["Predicho: no mora", "Predicho: mora"]
))

print("Métricas principales")
display(metricas)

print("Reporte de clasificación")
print(classification_report(y_test, y_pred, target_names=["no mora", "mora"]))


| P8 | Interpreta la matriz de confusión: ¿qué error sería más delicado para el negocio? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

| P9 | ¿Qué métrica observarías con más cuidado si el objetivo es detectar clientes que sí caerán en mora? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

## Actividad 6: Interpretación y predicción

In [ ]:
# Paso 6: interpretar variables importantes y visualizar el árbol
importancias = pd.DataFrame({
    "variable": variables_predictoras,
    "importancia": modelo_arbol.feature_importances_
}).sort_values("importancia", ascending=False)

display(importancias)

if HAS_PLOTS:
    plt.figure(figsize=(14, 7))
    plot_tree(
        modelo_arbol,
        feature_names=variables_predictoras,
        class_names=["no mora", "mora"],
        filled=True,
        rounded=True,
        fontsize=8
    )
    plt.title("Representación del árbol de decisión")
    plt.show()


In [ ]:
# Paso 7: clasificar un nuevo cliente
nuevo_cliente = pd.DataFrame([{
    "limite_credito": 50000,
    "edad": 35,
    "estado_pago_septiembre": 2,
    "estado_pago_agosto": 2,
    "estado_pago_julio": 0,
    "factura_septiembre": 45000,
    "pago_septiembre": 1000,
}])

clase_predicha = modelo_arbol.predict(nuevo_cliente)[0]
probabilidades = modelo_arbol.predict_proba(nuevo_cliente)[0]

resultado = "sí cae en mora" if clase_predicha == 1 else "no cae en mora"

print("Resultado predicho:", resultado)
print("Probabilidad de no mora:", round(probabilidades[0], 4))
print("Probabilidad de mora:", round(probabilidades[1], 4))
display(nuevo_cliente)


| P10 | Según el árbol y las variables importantes, ¿qué señales ayudan más a clasificar el riesgo de mora? |
|---|---|
| Respuesta |  |
|  |  |
|  |  |

## CONCLUSIONES

| 1. |  |
|---|---|
| 2. |  |
| 3. |  |

## MATERIAL COMPLEMENTARIO

- UCI Machine Learning Repository. Default of Credit Card Clients. https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients
- Kaggle. Default of Credit Card Clients Dataset. https://www.kaggle.com/datasets/uciml/default-of-credit-card-clients-dataset
- Scikit-learn documentation: DecisionTreeClassifier.